# Week 12: Data Engineering for AI

**Lab companion to [week_11.md](../agenda/week_11.md)**

**Objectives:**
- Practice SQL for AI data analysis
- Build a document processing pipeline
- Implement data quality checks
- Manage embedding versions

---

## Setup

In [ ]:
import sqlite3
import json
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Any, Optional
import hashlib

---
## Part 1: SQL for AI Data

SQL is essential for AI engineers - from extracting training data to analyzing RAG performance.

In [ ]:
# Create a sample database for RAG analytics
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Create tables
cursor.executescript("""
CREATE TABLE documents (
    id INTEGER PRIMARY KEY,
    filename TEXT,
    content TEXT,
    word_count INTEGER,
    created_at TIMESTAMP
);

CREATE TABLE chunks (
    id INTEGER PRIMARY KEY,
    document_id INTEGER,
    chunk_index INTEGER,
    chunk_text TEXT,
    FOREIGN KEY (document_id) REFERENCES documents(id)
);

CREATE TABLE search_logs (
    id INTEGER PRIMARY KEY,
    query TEXT,
    top_result_chunk_id INTEGER,
    similarity_score REAL,
    user_feedback TEXT,  -- 'helpful', 'not_helpful', NULL
    timestamp TIMESTAMP
);
""")

# Insert sample data
documents = [
    ("vacation_policy.txt", "Employees receive 20 days PTO per year. Request vacation 2 weeks in advance.", 12, "2024-01-15"),
    ("benefits.txt", "401k matching up to 4%. Health insurance covers 80% of premiums.", 11, "2024-01-20"),
    ("remote_work.txt", "Remote work allowed on Tuesdays and Thursdays. VPN required for all remote access.", 13, "2024-02-01"),
    ("onboarding.txt", "New employees complete orientation in first week. IT setup takes 1-2 days.", 12, "2024-02-10"),
]

cursor.executemany(
    "INSERT INTO documents (filename, content, word_count, created_at) VALUES (?, ?, ?, ?)",
    documents
)

chunks = [
    (1, 0, "Employees receive 20 days PTO per year."),
    (1, 1, "Request vacation 2 weeks in advance."),
    (2, 0, "401k matching up to 4%."),
    (2, 1, "Health insurance covers 80% of premiums."),
    (3, 0, "Remote work allowed on Tuesdays and Thursdays."),
    (3, 1, "VPN required for all remote access."),
    (4, 0, "New employees complete orientation in first week."),
    (4, 1, "IT setup takes 1-2 days."),
]

cursor.executemany(
    "INSERT INTO chunks (document_id, chunk_index, chunk_text) VALUES (?, ?, ?)",
    chunks
)

search_logs = [
    ("vacation days", 1, 0.92, "helpful", "2024-03-01 10:00:00"),
    ("401k", 3, 0.88, "helpful", "2024-03-01 10:05:00"),
    ("remote work policy", 5, 0.95, "helpful", "2024-03-01 10:10:00"),
    ("vacation policy", 1, 0.94, "helpful", "2024-03-01 11:00:00"),
    ("health benefits", 4, 0.85, "not_helpful", "2024-03-01 11:30:00"),
    ("PTO", 1, 0.91, "helpful", "2024-03-02 09:00:00"),
    ("work from home", 5, 0.89, "helpful", "2024-03-02 09:30:00"),
    ("new hire", 7, 0.82, None, "2024-03-02 10:00:00"),
]

cursor.executemany(
    "INSERT INTO search_logs (query, top_result_chunk_id, similarity_score, user_feedback, timestamp) VALUES (?, ?, ?, ?, ?)",
    search_logs
)

conn.commit()
print("✓ Database created with sample data")

### Practice 1.1: Basic JOINs

Write a query to show each chunk with its source document filename.

In [ ]:
# TODO: Write a JOIN query
query = """
SELECT
    d.filename,
    c.chunk_index,
    c.chunk_text
FROM chunks c
-- Your JOIN here
ORDER BY d.filename, c.chunk_index
"""

# Solution:
query = """
SELECT
    d.filename,
    c.chunk_index,
    c.chunk_text
FROM chunks c
JOIN documents d ON c.document_id = d.id
ORDER BY d.filename, c.chunk_index
"""

for row in cursor.execute(query):
    print(f"{row[0]} [{row[1]}]: {row[2][:50]}...")

### Practice 1.2: Aggregation and GROUP BY

Find documents with their chunk counts, ordered by most chunks first.

In [ ]:
# TODO: Write GROUP BY query
query = """
SELECT
    d.filename,
    COUNT(c.id) as chunk_count
FROM documents d
LEFT JOIN chunks c ON d.id = c.document_id
GROUP BY d.id
ORDER BY chunk_count DESC
"""

print("Documents by chunk count:")
for row in cursor.execute(query):
    print(f"  {row[0]}: {row[1]} chunks")

### Practice 1.3: CTEs (Common Table Expressions)

Find the most searched chunks using a CTE.

In [ ]:
# CTE example
query = """
WITH search_stats AS (
    SELECT
        top_result_chunk_id,
        COUNT(*) as search_count,
        AVG(similarity_score) as avg_score,
        SUM(CASE WHEN user_feedback = 'helpful' THEN 1 ELSE 0 END) as helpful_count
    FROM search_logs
    GROUP BY top_result_chunk_id
)
SELECT
    c.chunk_text,
    d.filename,
    ss.search_count,
    ROUND(ss.avg_score, 2) as avg_score,
    ss.helpful_count
FROM search_stats ss
JOIN chunks c ON ss.top_result_chunk_id = c.id
JOIN documents d ON c.document_id = d.id
ORDER BY ss.search_count DESC
LIMIT 5
"""

print("Most searched chunks:")
for row in cursor.execute(query):
    print(f"  '{row[0][:40]}...' from {row[1]}")
    print(f"    Searches: {row[2]}, Avg score: {row[3]}, Helpful: {row[4]}")

### Practice 1.4: Window Functions

Rank chunks by similarity score within each search query.

In [ ]:
# Note: SQLite supports window functions since 3.25
query = """
SELECT
    query,
    similarity_score,
    ROW_NUMBER() OVER (ORDER BY similarity_score DESC) as overall_rank
FROM search_logs
"""

print("Searches ranked by similarity:")
for row in cursor.execute(query):
    print(f"  #{row[2]}: '{row[0]}' (score: {row[1]})")

---
## Part 2: Data Pipeline

Build an ETL pipeline for processing documents.

In [ ]:
@dataclass
class PipelineStage:
    name: str
    started_at: datetime = None
    completed_at: datetime = None
    records_in: int = 0
    records_out: int = 0
    errors: List[str] = None

@dataclass
class Document:
    id: str
    filename: str
    content: str
    metadata: dict

class DocumentPipeline:
    """Simple ETL pipeline for documents."""

    def __init__(self):
        self.stages: List[PipelineStage] = []

    def _hash_content(self, content: str) -> str:
        return hashlib.md5(content.encode()).hexdigest()[:12]

    def extract(self, documents: List[dict]) -> List[Document]:
        """Extract: Load raw documents."""
        stage = PipelineStage(name="extract", started_at=datetime.now(), errors=[])

        result = []
        for doc in documents:
            try:
                result.append(Document(
                    id=self._hash_content(doc["content"]),
                    filename=doc["filename"],
                    content=doc["content"],
                    metadata={"source": "manual"}
                ))
            except Exception as e:
                stage.errors.append(str(e))

        stage.records_out = len(result)
        stage.completed_at = datetime.now()
        self.stages.append(stage)

        return result

    def transform(self, documents: List[Document]) -> List[Document]:
        """Transform: Clean and enrich."""
        stage = PipelineStage(
            name="transform",
            started_at=datetime.now(),
            records_in=len(documents),
            errors=[]
        )

        cleaned = []
        for doc in documents:
            # Clean content
            content = doc.content.strip()
            content = ' '.join(content.split())  # Normalize whitespace

            # Validation
            if len(content) < 10:
                stage.errors.append(f"{doc.filename}: Too short")
                continue

            doc.content = content
            doc.metadata["word_count"] = len(content.split())
            doc.metadata["char_count"] = len(content)
            cleaned.append(doc)

        stage.records_out = len(cleaned)
        stage.completed_at = datetime.now()
        self.stages.append(stage)

        return cleaned

    def load(self, documents: List[Document]) -> int:
        """Load: Store processed documents."""
        stage = PipelineStage(
            name="load",
            started_at=datetime.now(),
            records_in=len(documents),
            errors=[]
        )

        # In real pipeline, would save to database/files
        saved = len(documents)

        stage.records_out = saved
        stage.completed_at = datetime.now()
        self.stages.append(stage)

        return saved

    def run(self, raw_docs: List[dict]) -> dict:
        """Run the full ETL pipeline."""
        print("Running ETL pipeline...")

        docs = self.extract(raw_docs)
        print(f"  Extract: {len(docs)} documents")

        docs = self.transform(docs)
        print(f"  Transform: {len(docs)} documents")

        saved = self.load(docs)
        print(f"  Load: {saved} documents")

        return self.summary()

    def summary(self) -> dict:
        return {
            "stages": len(self.stages),
            "total_errors": sum(len(s.errors or []) for s in self.stages),
            "details": [
                {
                    "name": s.name,
                    "in": s.records_in,
                    "out": s.records_out,
                    "errors": len(s.errors or [])
                }
                for s in self.stages
            ]
        }

In [ ]:
# Test the pipeline
raw_documents = [
    {"filename": "doc1.txt", "content": "This is a valid document with enough content."},
    {"filename": "doc2.txt", "content": "Short"},  # Should be filtered
    {"filename": "doc3.txt", "content": "   Another document   with weird    whitespace   "},
]

pipeline = DocumentPipeline()
summary = pipeline.run(raw_documents)

print("\nPipeline summary:")
print(json.dumps(summary, indent=2))

---
## Part 3: Data Quality

Implement data quality checks for RAG datasets.

In [ ]:
from typing import Callable

@dataclass
class QualityRule:
    name: str
    column: str
    check: Callable[[Any], bool]
    severity: str = "error"

class DataQualityChecker:
    """Check data quality for AI datasets."""

    def __init__(self):
        self.rules: List[QualityRule] = []

    def add_rule(self, name: str, column: str, check: Callable, severity: str = "error"):
        self.rules.append(QualityRule(name, column, check, severity))
        return self

    def check(self, data: List[Dict]) -> Dict:
        """Run all quality checks."""
        results = []

        for rule in self.rules:
            failures = []
            for i, row in enumerate(data):
                value = row.get(rule.column)
                try:
                    if not rule.check(value):
                        failures.append(f"Row {i}: {rule.column}={repr(value)[:50]}")
                except Exception as e:
                    failures.append(f"Row {i}: Error - {e}")

            results.append({
                "rule": rule.name,
                "severity": rule.severity,
                "passed": len(failures) == 0,
                "failed_count": len(failures),
                "samples": failures[:3]
            })

        passed = sum(1 for r in results if r["passed"])

        return {
            "total_rules": len(results),
            "passed": passed,
            "failed": len(results) - passed,
            "pass_rate": round(passed / len(results) * 100, 1) if results else 100,
            "results": results
        }

In [ ]:
# Create checker for RAG data
checker = DataQualityChecker()

# Add rules
checker.add_rule(
    "text_not_empty",
    "text",
    lambda x: x is not None and len(str(x).strip()) > 0
)

checker.add_rule(
    "text_min_length",
    "text",
    lambda x: len(str(x)) >= 20 if x else False
)

checker.add_rule(
    "source_exists",
    "source",
    lambda x: x is not None and len(str(x)) > 0
)

checker.add_rule(
    "embedding_dimension",
    "embedding",
    lambda x: x is not None and len(x) == 1536,
    severity="warning"  # Warning, not error
)

In [ ]:
# Test with sample data
test_data = [
    {"text": "This is a valid chunk with enough content.",
     "embedding": [0.1] * 1536, "source": "doc1.txt"},
    {"text": "", "embedding": None, "source": ""},  # All bad
    {"text": "Another valid text chunk here.",
     "embedding": [0.2] * 1536, "source": "doc2.txt"},
    {"text": "Short", "embedding": [0.1] * 100, "source": "doc3.txt"},  # Wrong dim
]

results = checker.check(test_data)

print(f"Quality Check: {results['pass_rate']}% pass rate")
print(f"Passed: {results['passed']}/{results['total_rules']} rules\n")

for r in results["results"]:
    status = "✓" if r["passed"] else "✗"
    print(f"{status} {r['rule']} ({r['severity']}): {r['failed_count']} failures")
    for sample in r["samples"]:
        print(f"    - {sample}")

---
## Part 4: Embedding Version Management

Track embedding model versions to know when re-embedding is needed.

In [ ]:
@dataclass
class EmbeddingRecord:
    entity_id: str
    entity_type: str
    model_name: str
    model_version: str
    created_at: datetime
    # In real system, would also store the embedding vector

class EmbeddingManager:
    """Manage embedding versions."""

    def __init__(self):
        self.store: Dict[str, List[EmbeddingRecord]] = {}
        self.current_model = "text-embedding-3-small"
        self.current_version = "2024-01"

    def store_embedding(self, entity_id: str, entity_type: str) -> EmbeddingRecord:
        """Store a new embedding record."""
        record = EmbeddingRecord(
            entity_id=entity_id,
            entity_type=entity_type,
            model_name=self.current_model,
            model_version=self.current_version,
            created_at=datetime.now()
        )

        if entity_id not in self.store:
            self.store[entity_id] = []
        self.store[entity_id].append(record)

        return record

    def get_latest(self, entity_id: str) -> Optional[EmbeddingRecord]:
        """Get the most recent embedding for an entity."""
        versions = self.store.get(entity_id, [])
        if not versions:
            return None
        return max(versions, key=lambda x: x.created_at)

    def needs_update(self, entity_id: str) -> bool:
        """Check if entity needs re-embedding."""
        latest = self.get_latest(entity_id)
        if not latest:
            return True
        return latest.model_version != self.current_version

    def get_outdated(self) -> List[str]:
        """Get all entities that need re-embedding."""
        return [eid for eid in self.store if self.needs_update(eid)]

    def update_model_version(self, new_version: str) -> List[str]:
        """Update model version (triggers need for re-embedding)."""
        self.current_version = new_version
        outdated = self.get_outdated()
        print(f"Model updated to {new_version}. {len(outdated)} entities need re-embedding.")
        return outdated

In [ ]:
# Test embedding manager
manager = EmbeddingManager()

# Store some embeddings
manager.store_embedding("doc_001", "document")
manager.store_embedding("doc_002", "document")
manager.store_embedding("chunk_001", "chunk")

print(f"Current model version: {manager.current_version}")
print(f"Entities stored: {len(manager.store)}")
print(f"Any need update? {len(manager.get_outdated())} entities\n")

# Simulate model update
print("--- Upgrading model ---")
outdated = manager.update_model_version("2024-02")
print(f"Outdated entities: {outdated}")

# Re-embed one document
print("\n--- Re-embedding doc_001 ---")
manager.store_embedding("doc_001", "document")
print(f"doc_001 needs update: {manager.needs_update('doc_001')}")
print(f"doc_002 needs update: {manager.needs_update('doc_002')}")

---
## Exercises

1. **SQL Challenge:** Write a query to find the "retrieval success rate" - percentage of searches that got "helpful" feedback.

2. **Pipeline Extension:** Add a chunking stage to the pipeline that splits documents into 100-word chunks.

3. **Quality Rules:** Add a quality rule that checks if text contains any common boilerplate (e.g., "Lorem ipsum").

In [ ]:
# Exercise 1: SQL - Retrieval success rate
# Your query here

success_query = """
SELECT
    COUNT(*) as total_searches,
    SUM(CASE WHEN user_feedback = 'helpful' THEN 1 ELSE 0 END) as helpful,
    ROUND(
        100.0 * SUM(CASE WHEN user_feedback = 'helpful' THEN 1 ELSE 0 END) /
        COUNT(CASE WHEN user_feedback IS NOT NULL THEN 1 END),
        1
    ) as success_rate
FROM search_logs
"""

print("Retrieval success rate:")
for row in cursor.execute(success_query):
    print(f"  Total: {row[0]}, Helpful: {row[1]}, Success rate: {row[2]}%")

---
## Summary

In this lab you learned:
- **SQL for AI:** JOINs, CTEs, aggregations, window functions
- **ETL pipelines:** Extract, Transform, Load pattern
- **Data quality:** Rule-based validation checks
- **Embedding management:** Version tracking for re-embedding

These skills are essential for enterprise AI engineering!